# Geneformer Foundation Model vs Scanpy PCA: PBMC 3K

## Project objective

This notebook validates and explores a comparison between pretrained Geneformer V1 cell embeddings and a conventional Scanpy PCA representation.

**Important scope:** this notebook analyses saved, validated outputs. It does not rerun Geneformer tokenization, model inference, or the complete Scanpy preprocessing pipeline from raw counts.

## Dataset and representations

- Dataset: PBMC 3K
- Cells: 2,671
- Supplied cell-type groups: 6
- Geneformer V1: 256-dimensional mean-pooled final-layer embeddings
- Scanpy baseline: 2,000 highly variable genes and 50 PCA components
- Evaluation metric: cosine silhouette score

Cell-type annotations are preliminary and are not independent biological ground truth.


## 1. Imports and paths

Paths are resolved from the repository root when the notebook is launched there, or from the notebook's parent directory when launched inside `notebooks/`.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd()
if (cwd / "results" / "verified_comparison").exists():
    PROJECT_DIR = cwd
elif (cwd.parent / "results" / "verified_comparison").exists():
    PROJECT_DIR = cwd.parent
else:
    raise FileNotFoundError(
        "Cannot find results/verified_comparison. "
        "Run the notebook from the repository root or notebooks/."
    )

RESULTS_DIR = PROJECT_DIR / "results" / "verified_comparison"

print("Project directory:", PROJECT_DIR.resolve())
print("Comparison results:", RESULTS_DIR.resolve())


## 2. Load the validated comparison outputs

These CSV files were generated in the earlier analysis workflow. Loading them does not rerun the underlying model or preprocessing.


In [ ]:
embedding_df = pd.read_csv(RESULTS_DIR / "geneformer_embeddings_256d.csv")
pca_df = pd.read_csv(RESULTS_DIR / "scanpy_pca_scores_50d.csv")
silhouette_df = pd.read_csv(RESULTS_DIR / "per_cell_silhouette_scores.csv")
saved_type_summary = pd.read_csv(RESULTS_DIR / "cell_type_silhouette_summary.csv")
geneformer_umap = pd.read_csv(RESULTS_DIR / "geneformer_umap_coordinates.csv")
pca_umap = pd.read_csv(RESULTS_DIR / "scanpy_pca_umap_coordinates.csv")

with (RESULTS_DIR / "comparison_metrics.json").open(encoding="utf-8") as f:
    saved_metrics = json.load(f)

print("Geneformer embedding table:", embedding_df.shape)
print("Scanpy PCA table:", pca_df.shape)
print("Per-cell silhouette table:", silhouette_df.shape)
print("Cell-type summary:", saved_type_summary.shape)
display(pd.DataFrame([saved_metrics]).T.rename(columns={0: "value"}))


## 3. Validate dimensions, completeness, and cell identity

The checks below verify that both representations contain the expected number of cells and that the same barcodes and labels are represented. They also check that embedding values are finite.


In [ ]:
geneformer_features = [str(i) for i in range(256)]
pca_features = [f"PC{i}" for i in range(1, 51)]

assert len(embedding_df) == 2671, "Unexpected Geneformer cell count"
assert len(pca_df) == 2671, "Unexpected PCA cell count"
assert len(silhouette_df) == 2671, "Unexpected silhouette row count"

assert set(geneformer_features).issubset(embedding_df.columns)
assert set(pca_features).issubset(pca_df.columns)

assert embedding_df["cell_barcode"].is_unique, "Duplicate Geneformer barcodes"
assert pca_df["cell_barcode"].is_unique, "Duplicate PCA barcodes"
assert set(embedding_df["cell_barcode"]) == set(pca_df["cell_barcode"]), (
    "Geneformer and PCA barcode sets differ"
)

geneformer_labels = embedding_df.set_index("cell_barcode")[["leiden", "cell_type"]].sort_index()
pca_labels = pca_df.set_index("cell_barcode")[["leiden", "cell_type"]].sort_index()
geneformer_labels["leiden"] = geneformer_labels["leiden"].astype(str)
pca_labels["leiden"] = pca_labels["leiden"].astype(str)

pd.testing.assert_frame_equal(geneformer_labels, pca_labels)

X_geneformer = embedding_df[geneformer_features].to_numpy(dtype=float)
X_pca = pca_df[pca_features].to_numpy(dtype=float)

assert np.isfinite(X_geneformer).all(), "Non-finite Geneformer values"
assert np.isfinite(X_pca).all(), "Non-finite PCA values"
assert silhouette_df[["Geneformer", "Scanpy_PCA"]].notna().all().all()

print("Validation passed.")
print(f"Cells: {len(embedding_df):,}")
print(f"Geneformer dimensions: {X_geneformer.shape[1]}")
print(f"Scanpy PCA dimensions: {X_pca.shape[1]}")
print(f"Cell types: {embedding_df['cell_type'].nunique()}")
print("Barcode and label alignment: passed")


## 4. Recalculate overall silhouette metrics

Silhouette scores are loaded from the saved per-cell evaluation. This cell independently calculates their means and compares them with the recorded summary metrics.


In [ ]:
geneformer_mean = float(silhouette_df["Geneformer"].mean())
pca_mean = float(silhouette_df["Scanpy_PCA"].mean())
difference = geneformer_mean - pca_mean

recalculated_metrics = pd.DataFrame({
    "representation": ["Geneformer V1", "Scanpy PCA"],
    "dimensions": [256, 50],
    "cosine_silhouette": [geneformer_mean, pca_mean],
})

display(recalculated_metrics.round(4))

print(f"Geneformer minus PCA: {difference:.4f}")

assert np.isclose(
    geneformer_mean, saved_metrics["geneformer_overall_silhouette"], atol=5e-4
), "Geneformer score does not match saved summary"
assert np.isclose(
    pca_mean, saved_metrics["scanpy_pca_overall_silhouette"], atol=5e-4
), "PCA score does not match saved summary"
assert np.isclose(
    difference, saved_metrics["difference_geneformer_minus_pca"], atol=5e-4
), "Difference does not match saved summary"
print("Saved overall metrics match the recalculated means.")


## 5. Compare mean silhouette by cell type

Positive silhouette values generally indicate that a cell is closer to its assigned group than to other groups; negative values indicate possible overlap or assignment mismatch. These values evaluate separation relative to the supplied labels, not biological correctness.


In [ ]:
recalculated_by_type = (
    silhouette_df.groupby("cell_type", as_index=False)
    .agg(
        cells=("Geneformer", "size"),
        Geneformer_mean=("Geneformer", "mean"),
        PCA_mean=("Scanpy_PCA", "mean"),
        Geneformer_median=("Geneformer", "median"),
        PCA_median=("Scanpy_PCA", "median"),
    )
)
recalculated_by_type["Difference_Geneformer_minus_PCA"] = (
    recalculated_by_type["Geneformer_mean"] - recalculated_by_type["PCA_mean"]
)

display(
    recalculated_by_type.sort_values("cell_type").round(4)
)


## 6. Plot the saved UMAP comparison

The coordinates are loaded from the saved analysis. UMAP is used here for visualization only; the absolute coordinate values and orientations are not comparable across independently fitted UMAPs.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, frame, title in [
    (axes[0], geneformer_umap, "Geneformer V1 embeddings"),
    (axes[1], pca_umap, "Scanpy PCA representation"),
]:
    required = {"UMAP1", "UMAP2", "cell_type", "cell_barcode"}
    assert required.issubset(frame.columns)
    assert len(frame) == 2671
    assert frame["cell_barcode"].is_unique
    assert frame[["UMAP1", "UMAP2"]].notna().all().all()

    for cell_type, group in frame.groupby("cell_type", sort=True):
        ax.scatter(
            group["UMAP1"], group["UMAP2"],
            s=8, alpha=0.7, label=cell_type, rasterized=True
        )
    ax.set_title(title)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(
    handles, labels, title="Supplied cell type",
    loc="center left", bbox_to_anchor=(1.0, 0.5)
)
fig.tight_layout(rect=[0, 0, 0.82, 1])
plt.show()


## 7. Interpretation

For this PBMC dataset and these particular configurations, the saved cosine silhouette means are:

- Geneformer V1 (256 dimensions): approximately **0.1075**
- Scanpy PCA (50 dimensions): approximately **0.2098**
- Difference (Geneformer minus PCA): approximately **-0.1023**

The Scanpy PCA representation therefore separates the supplied labels more strongly under this metric in this experiment. This does not establish universal superiority of PCA over foundation-model embeddings.

### Limitations

- One dataset and one Geneformer configuration were evaluated.
- The labels are preliminary annotations, not independent ground truth.
- Rare groups have small sample sizes, so their group-specific scores are less stable.
- The two representations have different dimensionalities and construction methods.
- UMAP is a visualization, not a quantitative measure of representation quality.
- This notebook validates saved outputs; it does **not** reproduce tokenization, model inference, or the raw-count-to-PCA preprocessing from scratch.
- The earlier Geneformer tokenization workflow required a pandas compatibility patch. A clean, fully reproducible environment and end-to-end pipeline remain future work.

## 8. Output files

The source outputs are in `results/verified_comparison/`. This notebook does not overwrite those files.
